# Context Selection: Submodular Coverage, MMR, and Determinantal Point Processes

*The showpiece of the Information Theory of RAG layer — and the payoff of the arc PMI → noisy-channel → retrieval-vs-long-context.*

That arc ended on a verdict: stuffing the window does not improve the answer, because the extra passages
are **redundant** (PMI measured it in bits) or are same-sector distractors. The redundancy that flattened
the quality curve **is** the diminishing-marginal-returns property of a *submodular* set function. This
notebook imports the canonical module `context_selection_submodular_dpp.py` (which owns every number) and
walks the topic section by section. It never redefines the math; it calls into the module.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import context_selection_submodular_dpp as csel

corpus = csel._corpus()
print(f"candidate pools: {corpus['n_queries']} queries, {corpus['pool_size']} candidates each "
      f"({csel.N_GENERIC} sector-generic near-duplicates + 1 disambiguator + {csel.N_DISTRACT} distractors), "
      f"{corpus['K']} answer prototypes, d={corpus['dim']}")

## 1 · Diminishing returns, formalized: submodular coverage

The facility-location objective $f(S)=\sum_i w_i\max_{j\in S}\operatorname{sim}(i,j)$ rewards a selection for
*standing in for* the whole candidate pool. It is **monotone submodular**: the marginal value of adding a
passage to a larger set is no greater than adding it to a smaller one — exactly the saturation PMI measured.
Greedy's marginal gains diminish.

In [ ]:
wi = csel.worked_pool_index(corpus)
pool = corpus["pools"][wi]
cov = csel.coverage_curve(pool)
print("greedy coverage f(S_k):", [round(v, 3) for v in cov["values"]])
print("marginal gains       :", [round(v, 3) for v in cov["gains"]])
print("gains are non-increasing:", all(b <= a + 1e-9 for a, b in zip(cov["gains"][1:], cov["gains"][2:])))

## 2 · The greedy guarantee — Nemhauser–Wolsey–Fisher (1978)

Maximizing a monotone submodular function under a cardinality constraint is NP-hard, but the **greedy**
algorithm returns a set worth at least $\left(1-\tfrac1e\right)\approx 0.632$ of the optimum. On the smooth
finance pool greedy actually *reaches* OPT, so we exhibit the bound on a small constructed instance where
greedy is **strictly** suboptimal — yet still above the $0.632$ floor.

In [ ]:
wc = csel.nwf_worst_case()
for k, g, o, fl in zip(wc["ks"], wc["greedy"], wc["opt"], wc["floor"]):
    print(f"k={k}: greedy={g:.0f}  OPT={o:.0f}  floor(0.632*OPT)={fl:.2f}  ratio={g/o:.3f}")
print("a real gap appears at k=2 (greedy 9 < OPT 12), still well above the floor")

## 3 · MMR — the heuristic that does not earn the theorem

Maximal Marginal Relevance, $\lambda\,\operatorname{rel}(d,q)-(1-\lambda)\max_{d'\in S}\operatorname{sim}(d,d')$,
takes its penalty against the *evolving* chosen set, so it is not greedy on any fixed monotone submodular
objective and carries **no** $1-1/e$ guarantee. Here it selects a set with strictly lower coverage than the
greedy submodular set.

In [ ]:
sim = csel.cov_sim_matrix(pool["vecs"])
fval = csel._facility_fn(pool)
greedy_set = csel.greedy_select(fval, len(pool["vecs"]), csel.K_SELECT)["selected"]
mmr_set = csel.mmr_select(pool["rel"], sim, csel.K_SELECT, 0.3)["selected"]
print(f"greedy coverage = {fval(greedy_set):.3f}   MMR(lam=0.3) coverage = {fval(mmr_set):.3f}")
print("MMR under-covers greedy:", fval(mmr_set) < fval(greedy_set))

## 4 · Diversity as volume — determinantal point processes

A DPP puts $\mathcal P(S)\propto\det(L_S)$. With $L$ a Gram matrix, $\det(L_S)$ is the **squared volume** of
the parallelepiped the selected feature vectors span, and the quality $\times$ diversity factorization
$L=\operatorname{diag}(q)\,S\,\operatorname{diag}(q)$ gives $\det(L_S)=\bigl(\prod_{i\in S} q_i^2\bigr)\det(S_S)$.
Near-duplicates span a near-flat solid (near-zero volume) and are repelled. Exact MAP is NP-hard, but
$\log\det(I+L_S)$ is monotone submodular, so greedy MAP inherits the $1-1/e$ guarantee.

In [ ]:
quality = np.exp(csel.DPP_QSCALE * pool["rel"])
L = csel.dpp_kernel(quality, pool["vecs"])
S = csel.sim_matrix(pool["vecs"])
sel = [0, csel.N_GENERIC, csel.N_GENERIC + 1]   # a generic + the disambiguator + a distractor
det_L = np.linalg.det(L[np.ix_(sel, sel)])
qprod = np.prod(quality[sel] ** 2)
det_S = np.linalg.det(S[np.ix_(sel, sel)])
print(f"det(L_S) = {det_L:.4f}   (prod q^2) * det(S_S) = {qprod*det_S:.4f}   (quality x volume)")
print("factorization holds:", abs(det_L - qprod * det_S) < 1e-9)

## 5 · The honesty hinge — info gain is *not* submodular

The objective we actually want is the answer-information gain $I(A;D_S\mid Q)=H(A\mid Q)-H(A\mid Q,D_S)$.
But mutual information of a *set* with a target is **not submodular in general** — not even on this finance
corpus, and provably not for a synergistic XOR pair (each observation alone is useless; together they
determine the answer, so the marginal gain *increases* with conditioning). This is precisely why
facility-location coverage, not info gain, is the submodular backbone that earns the theorem.

In [ ]:
n = len(pool["vecs"])
fac_w = csel.submodularity_witness(csel._facility_fn(pool), n)
ig_w = csel.submodularity_witness(csel.info_gain_fn(corpus, pool, corpus["q"][wi]), n)
xor = csel.info_gain_xor_witness()
print(f"facility-location witness = {fac_w:+.4f}  (>= 0: submodular)")
print(f"info-gain witness         = {ig_w:+.4f}  (< 0: NOT submodular on this corpus)")
print(f"XOR: Delta(D2|empty)={xor['delta_empty']:.0f} bit, Delta(D2|{{D1}})={xor['delta_given_d1']:.0f} bit "
      f"-> gain INCREASES with conditioning (supermodular)")

## 6 · The payoff — diversity beats top-$k$ at a fixed budget

The most query-relevant passages are sector-generic near-duplicates that support the gold company *and* its
confusable peer equally; the **disambiguating** passage is less query-similar. So top-$k$ spends its budget
on the redundant cluster and leaves the answer split, while coverage- and diversity-aware selection reach
the disambiguator. We read every method's answer through the **imported** `answer_posterior_topk`, so the
numbers chain with the rest of the arc.

In [ ]:
selns = csel._method_selection(pool, csel.K_SELECT)
facets = {0: "generic", 1: "DISAMBIGUATOR", 3: "distractor"}
for m in csel.METHODS:
    fs = [facets[int(pool["facet"][i])] for i in selns[m]]
    print(f"{m:9s} picks {selns[m]} -> {fs}")
print()
pay = csel.selection_payoff(corpus)
for m in csel.METHODS:
    print(f"{m:9s}  Q(mass on truth) = {pay[m]['Q']:.3f}   coverage = {pay[m]['cover']:.3f}")
print("\ntop-k fills the budget with redundant generics; facility & DPP reach the disambiguator.")

## Verification — every claim is a test

The module's harness asserts each pedagogical claim: the collapse anchors (greedy@1 = argmax, MMR$(\lambda{=}1)$ =
top-$k$, the imported-readout anchor), the NWF guarantee and its visible gap, submodularity and diminishing
gains, MMR's missing guarantee, the DPP volume geometry and factorization, info gain's non-submodularity,
and the pinned payoff.

In [ ]:
csel._run_all()

## The numbers the lab mirrors

`viz_constants()` prints every value baked into `ContextSelectionLaboratory.tsx` to the decimal — the
viz $\leftrightarrow$ Python invariant. Change a number here, re-run, then update the `.tsx`; never the reverse.

In [ ]:
csel.viz_constants()